In [1]:
import os, sys
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(*args, **kwargs): pass

PROJECT_ROOT = os.path.abspath('..')
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'transit_sim'))

import model.gtfs_utils as gtfs_utils
from model.network import Network
from model.trains import Trains
from model.travelers import Travelers

np.random.seed(0)


## CONFIG — edit here to change scenario

In [2]:
# === INPUT CONFIGURATION — edit these to run a different scenario ===
LINE_NUM       = "6"             # line number or name
LINE_CODE      = f"bjl{LINE_NUM}"
SERVICE_ID     = "weekday"       # timetable service type
REMAP_STATION  = "jintai_lu"    # out-of-network exits remapped to this station

# Folders
RAW_DATA_DIR   = "raw_data"
INPUT_DIR      = "transit_sim_inputs"
OUTPUT_DIR     = "transit_sim_outputs"

# Input files
STATION_CODES_FILE = "station_codes.csv"              

# Simulation parameters
SIM_DATE          = '2026-06-24'  # date of simulation
TRAIN_CAPACITY    = 1960          # seats + standing per train
EXIT_WALKING_PP   = 90            # seconds, peak period
EXIT_WALKING_SP   = 90            # seconds, shoulder period
TRANSFER_TIME_PP  = 90            # seconds, peak period
TRANSFER_TIME_SP  = 90            # seconds, shoulder period
T_START_HOUR      = 5             # simulation window start (hour)
T_END_HOUR        = 12            # simulation window end (hour)
PEAK_START_HOUR   = 7.5           # peak window start (hour)
PEAK_END_HOUR     = 8.75          # peak window end (hour)
TIME_STEP         = 20            # timestep (seconds)
ROUTE_CHOICE_MODEL = "stops"      # select one: ["stops", "travel_time", "avg_travel_time"] 

# Derived — no need to edit
DEMAND_SCEN_NM = f"{LINE_CODE}_{SIM_DATE}"
OUTPUT_SCEN_NM = f"{LINE_CODE}_{SIM_DATE}_cap{TRAIN_CAPACITY}_wt{EXIT_WALKING_PP}-{EXIT_WALKING_SP}"
IN_PATH  = os.path.join(PROJECT_ROOT, INPUT_DIR)
OUT_PATH = os.path.join(PROJECT_ROOT, OUTPUT_DIR, OUTPUT_SCEN_NM)

for sub in ['train_outputs', 'traveler_outputs', 'link_volume']:
    os.makedirs(os.path.join(OUT_PATH, sub), exist_ok=True)

print(f'Output → {OUT_PATH}')

Output → /Users/suryabuchunde/Data/AAAM/transit_sim_outputs/bjl6_2026-06-24_cap1960_wt90-90


## Build network and schedule from GTFS

In [3]:
stop_times_file = os.path.join(IN_PATH, f'gtfs_{LINE_CODE}_stop_times.csv')
trips_file      = os.path.join(IN_PATH, f'gtfs_{LINE_CODE}_trips.csv')
stops_file      = os.path.join(IN_PATH, f'gtfs_{LINE_CODE}_stops.csv')

all_nodes, all_links, all_schedules = gtfs_utils.schedule_and_network_from_gtfs(
    stop_times_file, trips_file, stops_file,
    service_id=SERVICE_ID, train_capacity=TRAIN_CAPACITY
)

network = Network(all_nodes, all_links)
trains  = Trains(all_schedules, network)

network.all_nodes.to_csv(os.path.join(OUT_PATH, 'all_nodes.csv'), index=False)
network.all_links.to_csv(os.path.join(OUT_PATH, 'all_links.csv'), index=False)
trains.all_schedule.to_csv(os.path.join(OUT_PATH, 'all_schedule.csv'), index=False)

print(f'Nodes: {len(all_nodes)}, Links: {len(all_links)}')
display(network.all_nodes.head(2))

(350, 3)
(350, 4)


,route_id,trip_id,service_id,capacity
0,upward,12011,weekday,1960
1,upward,12056,weekday,1960
2,upward,12113,weekday,1960
3,upward,12140,weekday,1960
4,upward,22012,weekday,1960


stop_id ranges from 0 - 13
trip_id ranges from 1000 - 1349
Graph summary #edges: 110 #vertices: 42
Graph summary #edges: 110 #vertices: 42
Graph summary #edges: 110 #vertices: 42
Nodes: 42, Links: 110


/Users/suryabuchunde/Data/AAAM/transit_sim/model/gtfs_utils.py:90: RuntimeWarning: invalid value encountered in scalar subtract
  shift_factor = min(max(5, (maxx - minx)/10000), 50)


,route_stop_id,route_id,stop_id,stop_name,type,nid,stop_lon,stop_lat,geometry
0,upward-0,upward,0,jintai_lu,platform,0,inf,inf,POINT (Infinity Infinity)
1,upward-1,upward,1,shilipu,platform,1,inf,inf,POINT (Infinity Infinity)


## Load demand and find routes

In [4]:
od_file = os.path.join(IN_PATH, f'od_{DEMAND_SCEN_NM}.csv')
all_travelers_df = pd.read_csv(od_file)

# Derive NETWORK_STATIONS from the GTFS stop_times — this is the authoritative list
# of stations in the simulated network (timetable stations only, not all of station_codes.csv).
stop_times_df = pd.read_csv(stop_times_file)
NETWORK_STATIONS = set(stop_times_df['stop_id'].unique())

# Remap out-of-network exits to REMAP_STATION (western terminus of simulated segment).
# Passengers whose smartcard destination is not a Line 6 station are assumed to have
# ridden to jintai_lu before transferring.
all_travelers_df['out.station'] = np.where(
    all_travelers_df['out.station'].isin(NETWORK_STATIONS),
    all_travelers_df['out.station'],
    REMAP_STATION
)
# Drop same-station trips (origin == remapped destination).
# This removes passengers who boarded at jintai_lu and were heading further west —
# they are out of scope for the downward segment simulation.
all_travelers_df = all_travelers_df[
    all_travelers_df['in.station'] != all_travelers_df['out.station']
].copy()
# Drop trips whose origin is not in the simulated network
all_travelers_df = all_travelers_df[
    all_travelers_df['in.station'].isin(NETWORK_STATIONS)
].copy()
# Drop trips that depart after the simulation window (they would never board)
all_travelers_df = all_travelers_df[
    all_travelers_df['in.time'] < T_END_HOUR * 3600
].copy()

print(f'Total trips after filtering: {len(all_travelers_df):,}')
print(f"exit_type breakdown:\n{all_travelers_df['exit_type'].value_counts().to_string()}")

all_travelers_df['origin_nid']     = all_travelers_df['in.station'].map(network.station_nm_nid_dict)
all_travelers_df['destin_nid']     = all_travelers_df['out.station'].map(network.station_nm_nid_dict)
all_travelers_df['traveler_id']    = np.arange(len(all_travelers_df))
all_travelers_df['departure_time'] = all_travelers_df['in.time']
all_travelers_df = all_travelers_df[['traveler_id','origin_nid','destin_nid','departure_time']]

travelers = Travelers(all_travelers_df)
travelers.set_initial_status()

unfulfilled = travelers.find_routes(
    network,
    {"stops": network.network_g_stops, "travel_time": network.network_g_t, "avg_travel_time": network.network_g_tavg}[ROUTE_CHOICE_MODEL],
    trains,
    travelers.all_travelers['traveler_id'].values,
    routing_round_id=0
)
print(f'Travelers: {len(all_travelers_df):,}, unfulfilled: {unfulfilled}')

Total trips after filtering: 150,816
exit_type breakdown:
exit_type
1    83557
0    67259
Travelers: 150,816, unfulfilled: 0


## Run simulation

In [5]:
t_init     = T_START_HOUR * 3600
t_end      = T_END_HOUR   * 3600
peak_start = PEAK_START_HOUR * 3600
peak_end   = PEAK_END_HOUR   * 3600
status_change_list = []

for t in range(int(t_init), int(t_end), TIME_STEP):
    trains.update_location_occupancy(t, travelers.all_travelers)

    if peak_start <= t <= peak_end:
        sc, _, _ = travelers.traveler_update(
            network, trains, t,
            transfer_time=TRANSFER_TIME_PP,
            exit_walking_time=EXIT_WALKING_PP)
    else:
        sc, _, _ = travelers.traveler_update(
            network, trains, t,
            transfer_time=TRANSFER_TIME_SP,
            exit_walking_time=EXIT_WALKING_SP)

    status_change_list += sc

    # Save aggregated results every step
    trains.get_all_train_positions(network).to_csv(
        os.path.join(OUT_PATH, 'train_outputs', f'train_outputs_{t}.csv'), index=False)
    (
        travelers.all_travelers[['traveler_status','association']]
        .fillna('nan')
        .groupby(['traveler_status','association'])
        .size().to_frame('num_travelers').reset_index()
        .to_csv(os.path.join(OUT_PATH, 'traveler_outputs', f'agg_traveler_outputs_{t}.csv'), index=False)
    )

    if (t - t_init) % 300 == 0:
        print(f'Simulation at {t//3600}:{(t%3600)//60:02}  ({t}s)')
        if status_change_list:
            pd.concat(status_change_list).to_csv(
                os.path.join(OUT_PATH, 'traveler_outputs', f'status_change_{t}.csv'), index=False)
        status_change_list = []

travelers.all_travelers.to_csv(
    os.path.join(OUT_PATH, 'traveler_outputs', 'final_traveler_status.csv'), index=False)
travelers.all_key_nids.to_csv(
    os.path.join(OUT_PATH, 'traveler_outputs', '0_final_traveler_routes.csv'), index=False)
print(f'Done. Results in {OUT_PATH}')

Simulation at 5:00  (18000s)
Simulation at 5:05  (18300s)
Simulation at 5:10  (18600s)
Simulation at 5:15  (18900s)
Simulation at 5:20  (19200s)
Simulation at 5:25  (19500s)
Simulation at 5:30  (19800s)
Simulation at 5:35  (20100s)
Simulation at 5:40  (20400s)
Simulation at 5:45  (20700s)
Simulation at 5:50  (21000s)
Simulation at 5:55  (21300s)
Simulation at 6:00  (21600s)
Simulation at 6:05  (21900s)
Simulation at 6:10  (22200s)
Simulation at 6:15  (22500s)
Simulation at 6:20  (22800s)
Simulation at 6:25  (23100s)
Simulation at 6:30  (23400s)
Simulation at 6:35  (23700s)
Simulation at 6:40  (24000s)
Simulation at 6:45  (24300s)
Simulation at 6:50  (24600s)
Simulation at 6:55  (24900s)
Simulation at 7:00  (25200s)
Simulation at 7:05  (25500s)
Simulation at 7:10  (25800s)
Simulation at 7:15  (26100s)
Simulation at 7:20  (26400s)
Simulation at 7:25  (26700s)
Simulation at 7:30  (27000s)
Simulation at 7:35  (27300s)
Simulation at 7:40  (27600s)
Simulation at 7:45  (27900s)
Simulation at 